# Ridge Regression

## Review 1 — Regression Track

**Dataset:** Ames Housing  
**Target:** `SalePrice`  
**Notebook:** `02_ridge_regression.ipynb`

This notebook implements the **Ridge Regression** as one of the ten required regression algorithms.  
All regression notebooks use the same dataset, `random_state=42`, 80:20 held-out split, feature engineering, missing-value handling, and leakage-safe preprocessing so that model comparisons are fair.

> **Team responsibility:** After running the notebook, write your own observations and interpretation in the marked Markdown cells. The course guidelines require the team's analysis and interpretation to be original.

## 1. Imports and Project Paths

In [ ]:
from pathlib import Path
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Locate the project root whether Jupyter is started from the repository root
# or from Review-1/notebooks.
cwd = Path.cwd().resolve()
candidates = [cwd] + list(cwd.parents)
PROJECT_ROOT = next(
    (p for p in candidates if (p / "src" / "preprocessing.py").exists()),
    cwd
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from preprocessing import (
    RANDOM_STATE, TARGET, load_and_engineer, split_data,
    remove_training_outliers, make_preprocessor, make_polynomial_preprocessor,
    regression_metrics
)

DATA_PATH = PROJECT_ROOT / "Review-1" / "data" / "AmesHousing.csv"
RESULTS_DIR = PROJECT_ROOT / "Review-1" / "results"
MODELS_DIR = PROJECT_ROOT / "Review-1" / "models"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")

## 2. Load Dataset and Perform the Required Audit

We report dataset shape, data types, missing-value counts, duplicate rows, and the target distribution.

In [ ]:
df, X, y = load_and_engineer(DATA_PATH)

print(f"Dataset shape: {df.shape}")
display(df.head())

print("\nData types:")
display(df.dtypes.value_counts().rename_axis("dtype").reset_index(name="count"))

print("\nMissing values — top 20:")
display(df.isna().sum().sort_values(ascending=False).head(20).to_frame("missing_count"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget distribution:")
display(y.describe().to_frame("SalePrice"))

print("\nCategorical features:", len(X.select_dtypes(include="object").columns))
print("Numerical features:", len(X.select_dtypes(include=np.number).columns))

## 3. Exploratory Data Analysis

The following cells provide the Review 1 EDA requirements: feature distributions, target distribution, correlation heatmap, and feature-target scatter plots.

In [ ]:
# Distribution plots for all numerical features.
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
ncols = 4
nrows = int(np.ceil(len(numeric_cols) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows))
axes = np.asarray(axes).ravel()

for ax, col in zip(axes, numeric_cols):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(f"Distribution: {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")

for ax in axes[len(numeric_cols):]:
    ax.axis("off")

plt.suptitle("Numerical Feature Distributions", y=1.002, fontsize=16)
plt.tight_layout()
plt.show()

# Target distribution.
plt.figure(figsize=(9, 5))
sns.histplot(y, kde=True)
plt.title("SalePrice Distribution")
plt.xlabel("Sale Price")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# Correlation heatmap for numerical variables.
corr_cols = numeric_cols + [TARGET]
corr = df[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(18, 14))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap — Numerical Features and Target")
plt.tight_layout()
plt.show()

# Two required feature-target scatter plots.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=df, x="Overall Qual", y=TARGET, ax=axes[0], alpha=0.55)
axes[0].set_title("Overall Quality vs SalePrice")
axes[0].set_xlabel("Overall Quality")
axes[0].set_ylabel("Sale Price")

sns.scatterplot(data=df, x="Gr Liv Area", y=TARGET, ax=axes[1], alpha=0.55)
axes[1].set_title("Above-Ground Living Area vs SalePrice")
axes[1].set_xlabel("Gr Liv Area")
axes[1].set_ylabel("Sale Price")

plt.tight_layout()
plt.show()

### EDA — Team Observation

**Write your own observations here after inspecting the plots.**

Address:
- important distributions or skewness,
- notable correlations,
- relationships between the target and important features,
- any visible outliers or unusual patterns.

## 4. Preprocessing and Feature Engineering

### Engineered features

Two predictor-only features are created:

- `TotalSF` = basement + first-floor + second-floor area.
- `TotalBathrooms` = full bathrooms + half bathrooms + basement bathrooms, with half bathrooms counted as 0.5.

Identifier columns (`Order` and `PID`) are excluded because they identify records rather than represent meaningful property characteristics.

The train/test split is created before data-driven imputation, encoding, scaling, or outlier threshold estimation.

In [ ]:
# Train/test split is performed before any data-driven preprocessing.
X_train, X_test, y_train, y_test = split_data(X, y)

print("Before outlier treatment:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

# Check and treat extreme Gr Liv Area outliers using a threshold learned
# from the training set only. The held-out test set is never filtered.
X_train, y_train, outlier_threshold = remove_training_outliers(
    X_train, y_train, feature="Gr Liv Area", multiplier=3.0
)

print(f"\nTraining outlier threshold for Gr Liv Area: {outlier_threshold:.2f}")
print("After outlier treatment:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nTarget means:")
print("Train:", y_train.mean())
print("Test :", y_test.mean())

### Preprocessing Strategy

- Numerical missing values → median imputation.
- Categorical missing values → most-frequent imputation.
- Categorical variables → one-hot encoding with unknown-category protection.
- Scale-sensitive algorithms → numerical standardization fitted on training data only.
- Tree-based algorithms → no feature scaling required.
- Extreme `Gr Liv Area` values are checked and training-set extremes above a 3×IQR upper threshold are removed; the threshold is learned only from the training set.

## 5. Train Ridge Regression

The model is trained only on the training split. The held-out test set is used only for final evaluation.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

MODEL_NAME = "Ridge Regression"
RESULT_FILE = "ridge_regression"

base_model = Pipeline([
    ("preprocess", make_preprocessor(X_train, scale_numeric=True)),
    ("model", Ridge())
])

# Required hyperparameter tuning example.
param_grid = {
    "model__alpha": [0.1, 1.0, 10.0, 50.0, 100.0]
}
grid = GridSearchCV(base_model, param_grid=param_grid, scoring="r2", cv=5, n_jobs=-1)
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV R2:", grid.best_score_)

model = grid.best_estimator_
y_pred = model.predict(X_test)

## 6. Test-Set Evaluation

The mandatory regression metrics are **R², RMSE, and MAE**.

In [ ]:
metrics = regression_metrics(y_test, y_pred)
metrics_df = pd.DataFrame([metrics], index=[MODEL_NAME])
display(metrics_df)

# Save a machine-readable result for the comparison notebook.
result_path = RESULTS_DIR / f"{RESULT_FILE}.csv"
metrics_df.reset_index(names="Model").to_csv(result_path, index=False)

print(f"Saved results to: {result_path}")

### Coefficient inspection

For linear models, coefficients can be inspected to understand which transformed features have the strongest positive or negative association with the prediction. Interpretations must be written by the project team after reviewing the actual fitted coefficients.

In [ ]:
# Show the 20 largest absolute coefficients when the final estimator exposes them.
estimator = model.named_steps["model"]
if hasattr(estimator, "coef_"):
    feature_names = model.named_steps["preprocess"].get_feature_names_out()
    coef = np.ravel(estimator.coef_)
    coef_df = pd.DataFrame({"Feature": feature_names, "Coefficient": coef})
    coef_df["AbsoluteCoefficient"] = coef_df["Coefficient"].abs()
    display(coef_df.sort_values("AbsoluteCoefficient", ascending=False).head(20))

## 8. Prediction Diagnostics

In [ ]:
# Predicted vs actual
plt.figure(figsize=(7, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.65)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, linestyle="--", label="Ideal: Actual = Predicted")
plt.title(f"{MODEL_NAME} — Predicted vs Actual")
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")
plt.legend()
plt.tight_layout()
plt.show()

# Residual plot
residuals = y_test - y_pred
plt.figure(figsize=(8, 5))
sns.scatterplot(x=y_pred, y=residuals, alpha=0.65)
plt.axhline(0, linestyle="--", label="Zero Residual")
plt.title(f"{MODEL_NAME} — Residual Plot")
plt.xlabel("Predicted SalePrice")
plt.ylabel("Residual (Actual - Predicted)")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Cross-Validation Note

The course requires 5-fold cross-validated R² for the **two best-performing models**. After the common comparison table identifies those two models, run 5-fold CV for them and record the results in the final comparison documentation.

## 10. Team Interpretation

**Write the team's own interpretation here.**

Discuss:
- whether the model generalizes well,
- what the metric values mean,
- important strengths/limitations,
- whether scaling or regularization was useful,
- and how this model compares with the other algorithms after the common comparison is completed.

In [ ]:
import joblib

model_path = MODELS_DIR / f"{RESULT_FILE}.joblib"
joblib.dump(model, model_path)
print(f"Saved fitted model to: {model_path}")

## Notebook Completion Checklist

- [ ] Run all cells top-to-bottom.
- [ ] Verify there are no execution errors.
- [ ] Verify plots have titles and labelled axes.
- [ ] Verify the result CSV was generated.
- [ ] Verify the fitted model file was generated.
- [ ] Replace the interpretation placeholder with the team's own observations.